In [12]:
import os
import glob
from docx import Document
import pathlib
import textwrap
import google.generativeai as genai
import time

def read_docx(file_path):
    doc = Document(file_path)
    lines = []
    for para in doc.paragraphs:
        para_lines = para.text.split('\n')
        lines.extend(para_lines)
    return lines


def extract_subject_from_filename(filename):
    if 'Maths' in filename:
        return 'Maths Statement or Equation'
    elif 'Chemistry' in filename:
        return 'Chemistry Statement'
    elif 'Physics' in filename:
        return 'Physics Statement'
    elif 'General' in filename:
        return 'General Statement'


def generate_question(scenario,subject, model):
    prompt = f"""Imagine you are a human, this is the first time you are coming across this {subject}, you have no previous knowledge of it "{scenario}", what are the top 5 questions that would pop up in your head which would be most useful in learning about it as you are new to it. Give me a simple bullet point list, don't explain them or expand them."""
    try:
        response = model.generate_content(prompt)
        if hasattr(response, 'text') and response.text:
            # print(prompt+response.text)
            return prompt + response.text
        else:
            return prompt + "No response or unexpected format received."
    except Exception as e:
        print(f"An error occurred: {e}")
        return prompt+"Error generating question for this scenario."

    
def write_questions_to_file(questions, output_file_path):
    with open(output_file_path, "w", encoding="utf-8") as file:
        for question in questions:
            file.write(question + "\n\n")

            
def process_directory(input_directory, output_directory, model):
    os.makedirs(output_directory, exist_ok=True)
    
    request_count = 0  # Initialize request counter

    for file_path in glob.glob(os.path.join(input_directory, '*.docx')):
        subject = extract_subject_from_filename(os.path.basename(file_path))

        lines = read_docx(file_path)
        questions = []
        for line in lines:
            questions.append(generate_question(line, subject, model))
            request_count += 1

            # Check if the request count has reached 50, if so, sleep for 30 seconds and reset the counter
            if request_count >= 50:
                print("Pausing for 30 seconds to avoid overloading the API...")
                time.sleep(30)  # Pause execution for 30 seconds
                request_count = 0  # Reset request counter after pausing

        output_file_name = os.path.basename(file_path).replace('.docx', '_results.txt')
        output_file_path = os.path.join(output_directory, output_file_name)
        
        write_questions_to_file(questions, output_file_path)
        print(f"Processed {file_path}, results written to {output_file_path}")


# Configuration and Model Initialization
GOOGLE_API_KEY = ""  # Replace with your actual Google API key
genai.configure(api_key=GOOGLE_API_KEY)


generation_config = {
  "temperature": 0.1,
  "top_p": 1,
  "top_k": 1,
  "max_output_tokens": 128,
}

safety_settings = [
  {
    "category": "HARM_CATEGORY_HARASSMENT",
    "threshold": "BLOCK_NONE"
  },
  {
    "category": "HARM_CATEGORY_HATE_SPEECH",
    "threshold": "BLOCK_NONE"
  },
  {
    "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
    "threshold": "BLOCK_NONE"
  },
  {
    "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
    "threshold": "BLOCK_NONE"
  }
]

model = genai.GenerativeModel('models/gemini-1.0-pro-001', safety_settings=safety_settings)

# Example Usage
process_directory('Question_data', 'outputs_gemini', model)


Pausing for 30 seconds to avoid overloading the API...
Pausing for 30 seconds to avoid overloading the API...
Pausing for 30 seconds to avoid overloading the API...
Processed Question_data/Chemistry - Intermediate.docx, results written to outputs_gemini/Chemistry - Intermediate_results.txt
Pausing for 30 seconds to avoid overloading the API...
Pausing for 30 seconds to avoid overloading the API...
Pausing for 30 seconds to avoid overloading the API...
Processed Question_data/Chemistry BASIC.docx, results written to outputs_gemini/Chemistry BASIC_results.txt
Pausing for 30 seconds to avoid overloading the API...
Pausing for 30 seconds to avoid overloading the API...
Pausing for 30 seconds to avoid overloading the API...
Processed Question_data/Chemsitry - Advanced.docx, results written to outputs_gemini/Chemsitry - Advanced_results.txt
Pausing for 30 seconds to avoid overloading the API...
Pausing for 30 seconds to avoid overloading the API...
Pausing for 30 seconds to avoid overloading